## Analyse Exploratoire (EDA) des données

L'EDA a pour but connaître les données avant de faire quoi que ce soit de sophistiqué. L'objectif est de répondre à des questions simples mais fondamentales sur le dataset GDELT Bénin et d'en tiré les premières statistiques utiles.

## 0. Imports

In [5]:
import pandas as pd
import numpy as np
from pathlib import Path
import re
import sys

## 1. Chargement des données

> Le chemin est relatif à la racine du repo.  
> Si l'exécution est sur **Google Colab**, remplacez `DATA_PATH` par le chemin Google Drive.

In [2]:
# Adapter selon votre environnement

# Option A : Local (VS Code, Jupyter classique)
EVENTS_DATA_PATH = Path('..') / 'data' / 'raw' / 'gdelt_bn_2025.csv'
GKG_DATA_PATH = Path('..') / 'data' / 'raw' / 'gdelt_gkg_bn_2025.csv'
# Option B : Google Colab (décommenter si nécessaire)
# from google.colab import drive
# drive.mount('/content/drive')
# EVENTS_DATA_PATH = '/content/drive/MyDrive/hackathon/gdelt_bn_2025.csv'
# GKG_DATA_PATH = '/content/drive/MyDrive/hackathon/gdelt_gkg_bn_2025.csv'

events_df = pd.read_csv(EVENTS_DATA_PATH, low_memory=False)
gkg_df = pd.read_csv(GKG_DATA_PATH, low_memory=False)

print(f'✅ Events dataset chargé : {events_df.shape[0]:,} lignes × {events_df.shape[1]} colonnes')
print(f'✅ GKG dataset chargé : {gkg_df.shape[0]:,} lignes × {gkg_df.shape[1]} colonnes')

✅ Events dataset chargé : 27,317 lignes × 61 colonnes
✅ GKG dataset chargé : 24,453 lignes × 8 colonnes


## 2. Structure du dataset

In [6]:
# 2.1 Aperçu général 
print('=' * 60)
print('APERÇU GÉNÉRAL Evénements')
print('=' * 60)
print(f"Nombre d'événements  : {events_df.shape[0]:,}")
print(f"Nombre de colonnes   : {events_df.shape[1]}")
print(f"Période couverte     : {events_df['SQLDATE'].min()} → {events_df['SQLDATE'].max()}")
print(f"Mémoire utilisée     : {events_df.memory_usage(deep=True).sum() / 1e6:.1f} MB")

print('\n' + '=' * 60)
print('APERÇU GÉNÉRAL GKG')
print('=' * 60)
print(f"Nombre de documents  : {gkg_df.shape[0]:,}")
print(f"Nombre de colonnes   : {gkg_df.shape[1]}")
print(f"Période couverte     : {gkg_df['Date'].min()} → {gkg_df['Date'].max()}")
print(f"Mémoire utilisée     : {gkg_df.memory_usage(deep=True).sum() / 1e6:.1f} MB")

APERÇU GÉNÉRAL Evénements
Nombre d'événements  : 27,317
Nombre de colonnes   : 61
Période couverte     : 20250101 → 20251231
Mémoire utilisée     : 17.9 MB

APERÇU GÉNÉRAL GKG
Nombre de documents  : 24,453
Nombre de colonnes   : 7
Période couverte     : 20250101003000 → 20251231201500
Mémoire utilisée     : 12.2 MB


In [7]:
# 2.2 Types de colonnes
print('\nTYPES DE COLONNES Evénements')
print('-' * 40)
print(events_df.dtypes.to_string())



TYPES DE COLONNES Evénements
----------------------------------------
GLOBALEVENTID              int64
SQLDATE                    int64
MonthYear                  int64
Year                       int64
FractionDate             float64
Actor1Code                   str
Actor1Name                   str
Actor1CountryCode            str
Actor1KnownGroupCode         str
Actor1EthnicCode             str
Actor1Religion1Code          str
Actor1Religion2Code          str
Actor1Type1Code              str
Actor1Type2Code              str
Actor1Type3Code              str
Actor2Code                   str
Actor2Name                   str
Actor2CountryCode            str
Actor2KnownGroupCode         str
Actor2EthnicCode             str
Actor2Religion1Code          str
Actor2Religion2Code          str
Actor2Type1Code              str
Actor2Type2Code              str
Actor2Type3Code              str
IsRootEvent                int64
EventCode                  int64
EventBaseCode              int64
Event

In [8]:
print('\n' + '-' * 40)
print('TYPES DE COLONNES GKG')
print('-' * 40)
print(gkg_df.dtypes.to_string())


----------------------------------------
TYPES DE COLONNES GKG
----------------------------------------
Date                int64
SourceCommonName      str
Persons               str
Organizations         str
Locations             str
Counts                str
TranslationInfo       str


In [3]:
# 2.3 Valeurs manquantes Evénements
missing = events_df.isnull().sum()
missing_pct = (missing / len(events_df) * 100).round(1)

missing_df = pd.DataFrame({
    'Valeurs manquantes': missing,
    '% manquant': missing_pct
}).query('`Valeurs manquantes` > 0').sort_values('% manquant', ascending=False)

print('\nCOLONNES AVEC VALEURS MANQUANTES - ÉVÉNEMENTS')
print('-' * 40)
print(missing_df.to_string())



COLONNES AVEC VALEURS MANQUANTES - ÉVÉNEMENTS
----------------------------------------
                       Valeurs manquantes  % manquant
Actor2Type3Code                     27310       100.0
Actor1Type3Code                     27299        99.9
Actor1Religion2Code                 27269        99.8
Actor2Religion2Code                 27275        99.8
Actor1EthnicCode                    27232        99.7
Actor2EthnicCode                    27230        99.7
Actor2Religion1Code                 27142        99.4
Actor1Religion1Code                 27145        99.4
Actor2KnownGroupCode                27048        99.0
Actor2Type2Code                     26852        98.3
Actor1KnownGroupCode                26822        98.2
Actor1Type2Code                     26519        97.1
Actor2Geo_ADM2Code                  24864        91.0
ActionGeo_ADM2Code                  24304        89.0
Actor1Geo_ADM2Code                  24302        89.0
Actor2Type1Code                     18641       

In [4]:
## 2.3 Valeurs manquantes GKG
missing = gkg_df.isnull().sum()
missing_pct = (missing / len(gkg_df) * 100).round(1)
missing_df = pd.DataFrame({
    'Valeurs manquantes': missing,
    '% manquant': missing_pct
}).query('`Valeurs manquantes` > 0').sort_values('% manquant', ascending=False)
print('\nCOLONNES AVEC VALEURS MANQUANTES - GKG')
print('-' * 40)
print(missing_df.to_string())


COLONNES AVEC VALEURS MANQUANTES - GKG
----------------------------------------
                 Valeurs manquantes  % manquant
TranslationInfo               20034        81.9
Counts                        18829        77.0
Persons                        3641        14.9
Organizations                  3173        13.0
Locations                        12         0.0


# **Nettoyage des données**

In [5]:
# Supprimer les colonnes d'events_df ayant > 80% de valeurs manquantes
thresh = 0.8
missing_pct_events = events_df.isnull().mean()
cols_to_drop = missing_pct_events[missing_pct_events > thresh].index.tolist()

if not cols_to_drop:
    print("Aucune colonne à supprimer (aucune > 80% manquante).")
else:
    print(f"Colonnes supprimées ({len(cols_to_drop)}): {cols_to_drop}")
    events_df_no_missing = events_df.drop(columns=cols_to_drop, inplace=False)
    print(f"Nouveau shape events_df: {events_df_no_missing.shape}")
    print(f"Colonnes restantes: {events_df_no_missing.columns.tolist()}")
    print(f"Nombre de colonnes restantes: {events_df_no_missing.shape[1]}")

Colonnes supprimées (15): ['Actor1KnownGroupCode', 'Actor1EthnicCode', 'Actor1Religion1Code', 'Actor1Religion2Code', 'Actor1Type2Code', 'Actor1Type3Code', 'Actor2KnownGroupCode', 'Actor2EthnicCode', 'Actor2Religion1Code', 'Actor2Religion2Code', 'Actor2Type2Code', 'Actor2Type3Code', 'Actor1Geo_ADM2Code', 'Actor2Geo_ADM2Code', 'ActionGeo_ADM2Code']
Nouveau shape events_df: (27317, 46)
Colonnes restantes: ['GLOBALEVENTID', 'SQLDATE', 'MonthYear', 'Year', 'FractionDate', 'Actor1Code', 'Actor1Name', 'Actor1CountryCode', 'Actor1Type1Code', 'Actor2Code', 'Actor2Name', 'Actor2CountryCode', 'Actor2Type1Code', 'IsRootEvent', 'EventCode', 'EventBaseCode', 'EventRootCode', 'QuadClass', 'GoldsteinScale', 'NumMentions', 'NumSources', 'NumArticles', 'AvgTone', 'Actor1Geo_Type', 'Actor1Geo_FullName', 'Actor1Geo_CountryCode', 'Actor1Geo_ADM1Code', 'Actor1Geo_Lat', 'Actor1Geo_Long', 'Actor1Geo_FeatureID', 'Actor2Geo_Type', 'Actor2Geo_FullName', 'Actor2Geo_CountryCode', 'Actor2Geo_ADM1Code', 'Actor2Geo_L

In [12]:
gkg_df.head()

,Date,SourceCommonName,Persons,Organizations,Locations,Counts,TranslationInfo
0,20251012024500,myjoyonline.com,NaN,ghana meteorological agency gmet,1#Nigeria#NI#NI#10#8#NI;1#Benin#BN#BN#9.5#2.25#BN,NaN,NaN
1,20251012030000,thenationonlineng.net,gloria bakare;ebonny musik,starburg empowerment foundation;university of ...,1#Benin#BN#BN#9.5#2.25#BN;1#Nigeria#NI#NI#10#8...,NaN,NaN
2,20251012030000,thenationonlineng.net,gloria bakare;ebonny musik,starburg empowerment foundation;university of ...,1#Benin#BN#BN#9.5#2.25#BN;1#Nigeria#NI#NI#10#8...,NaN,NaN
3,20251012190000,jagran.com,sivan a sun sharma;ashok mahato;gazipur a chau...,visa the company a operations;visa;group the c...,"4#Patna, Bihar, India#IN#IN34#25.6#85.1167#-21...",NaN,srclc:hin;eng:GT-HIN 1.0
4,20251012220000,thecable.ng,frank azuekor;bayo onanuga;mike ozekhome;bola ...,national open university;department of state s...,"1#Nigeria#NI#NI#10#8#NI;4#Ethiope, Delta, Nige...",KIDNAP#4#police officers#1#Nigeria#NI#NI#10#8#...,NaN


In [ ]:
# s'assurer que le code d'importation des modules est exexuté

# Importer la fonction depuis scripts/data_pipeline.py
sys.path.append(str(Path().resolve().parent))
import scripts.data_pipeline as dp
import importlib
# Exécute le fichier data_pipeline.py et le recharge afin de s’assurer que les dernières modifications sont prises en compte dans la session en cours.
importlib.reload(dp)

# Travailler sur une copie dédiée pour ne pas modifier le dataset brut
gkg_df_processing = gkg_df.copy(deep=True)

# Extraction des deux colonnes utiles
parsed = gkg_df_processing["TranslationInfo"].apply(dp.parse_translation_info).apply(pd.Series)
gkg_df_processing = pd.concat([gkg_df_processing, parsed], axis=1)

# Aperçu rapide
print(gkg_df_processing[["TranslationInfo", "translation_source_langs"]].head(30).to_string())


             TranslationInfo translation_source_langs
0   srclc:hin;eng:GT-HIN 1.0                      hin
1                        NaN                      NaN
2                        NaN                      NaN
3                        NaN                      NaN
4                        NaN                      NaN
5                        NaN                      NaN
6                        NaN                      NaN
7                        NaN                      NaN
8   srclc:spa;eng:GT-SPA 1.0                      spa
9                        NaN                      NaN
10                       NaN                      NaN
11                       NaN                      NaN
12                       NaN                      NaN
13                       NaN                      NaN
14                       NaN                      NaN
15                       NaN                      NaN
16  srclc:fra;eng:GT-FRA 1.0                      fra
17  srclc:fra;eng:GT-FRA 1.0

In [6]:
gkg_df_processing.head()

,Date,SourceCommonName,Persons,Organizations,Locations,Counts,V2Tone,TranslationInfo,translation_source_langs
0,20251012190000,jagran.com,sivan a sun sharma;ashok mahato;gazipur a chau...,visa the company a operations;visa;group the c...,"4#Patna, Bihar, India#IN#IN34#25.6#85.1167#-21...",NaN,"-4.30622009569378,1.19617224880383,5.502392344...",srclc:hin;eng:GT-HIN 1.0,hin
1,20251012220000,thecable.ng,frank azuekor;bayo onanuga;mike ozekhome;bola ...,national open university;department of state s...,"1#Nigeria#NI#NI#10#8#NI;4#Ethiope, Delta, Nige...",KIDNAP#4#police officers#1#Nigeria#NI#NI#10#8#...,"-7.40740740740741,1.97530864197531,9.382716049...",NaN,NaN
2,20251012220000,thecable.ng,frank azuekor;bayo onanuga;mike ozekhome;bola ...,national open university;department of state s...,"1#Nigeria#NI#NI#10#8#NI;4#Ethiope, Delta, Nige...",KIDNAP#4#police officers#1#Nigeria#NI#NI#10#8#...,"-7.40740740740741,1.97530864197531,9.382716049...",NaN,NaN
3,20251012220000,thecable.ng,frank azuekor;bayo onanuga;mike ozekhome;bola ...,national open university;department of state s...,"1#Nigeria#NI#NI#10#8#NI;4#Ethiope, Delta, Nige...",KIDNAP#4#police officers#1#Nigeria#NI#NI#10#8#...,"-7.40740740740741,1.97530864197531,9.382716049...",NaN,NaN
4,20251012220000,thecable.ng,frank azuekor;bayo onanuga;mike ozekhome;bola ...,national open university;department of state s...,"1#Nigeria#NI#NI#10#8#NI;4#Ethiope, Delta, Nige...",KIDNAP#4#police officers#1#Nigeria#NI#NI#10#8#...,"-7.40740740740741,1.97530864197531,9.382716049...",NaN,NaN


## Enregistrement des datasets

In [ ]:
# Définir les chemins de sortie
output_dir = Path('..') / 'data' / 'processed'
output_dir.mkdir(parents=True, exist_ok=True)

events_output = output_dir / 'events_cleaned.csv'
gkg_output = output_dir / 'gkg_cleaned.csv'

# Enregistrer events_df nettoyé
events_df_no_missing.to_csv(events_output, index=False)
print(f"✅ Events dataset enregistré : {events_output}")
print(f"   {events_df_no_missing.shape[0]:,} lignes × {events_df_no_missing.shape[1]} colonnes")

# Enregistrer gkg_df_processing avec les colonnes de traduction
gkg_df_processing.to_csv(gkg_output, index=False)
print(f"\n✅ GKG dataset (traité) enregistré : {gkg_output}")
print(f"   {gkg_df_processing.shape[0]:,} lignes × {gkg_df_processing.shape[1]} colonnes")
print(f"   Colonnes ajoutées: translation_source_langs")


✅ GKG dataset (traité) enregistré : ..\data\processed\gkg_cleaned.csv
   24,453 lignes × 9 colonnes
   Colonnes ajoutées: translation_source_langs


## Cleaning des données pour la période 2021-2026

In [2]:
# Adapter selon votre environnement

# Option A : Local (VS Code, Jupyter classique)
#THEMES_DATA_PATH = Path('..') / 'data' / 'raw' / 'gkg_themes_2021_2026.csv'
GKG_2021_2026_DATA_PATH = Path('..') / 'data' / 'raw' / 'gkg_2021_2026.csv'
EVENTS_DATA_PATH = Path('..') / 'data' / 'raw' / 'events_2021_2026.csv'
# Option B : Google Colab (décommenter si nécessaire)
# from google.colab import drive
# drive.mount('/content/drive')
# THEMES_DATA_PATH = '/content/drive/MyDrive/hackathon/gkg_themes_2021_2026.csv'
# GKG_2021_2026_DATA_PATH = '/content/drive/MyDrive/hackathon/gkg_2021_2026.csv'
# EVENTS_DATA_PATH = '/content/drive/MyDrive/hackathon/events_2021_2026.csv'

#events_df = pd.read_csv(EVENTS_DATA_PATH, low_memory=False)
#gkg_df = pd.read_csv(GKG_2021_2026_DATA_PATH, low_memory=False)
#themes_df = pd.read_csv(THEMES_DATA_PATH, low_memory=False)

## Merge of the datasets
#gkg_df = gkg_df.merge(themes_df, how='left', on='GKGRECORDID')

#print(f'✅ Events dataset chargé : {events_df.shape[0]:,} lignes × {events_df.shape[1]} colonnes')
#print(f'✅ GKG dataset chargé : {gkg_df.shape[0]:,} lignes × {gkg_df.shape[1]} colonnes')

In [ ]:
## Selection des données de GKG relatives au Bénin
def check_geographic_precision(locations_cell):
    """
    Analyse la colonne 'Locations' pour trouver le Bénin
    avec la précision de votre requête SQL (Type=1, Nom=benin, Code=BN).
    """
    if pd.isna(locations_cell) or locations_cell == "":
        return False

    # Premier niveau : séparation des blocs de lieux [16, 58]
    blocks = str(locations_cell).split(';')

    for block in blocks:
        # Second niveau : séparation des attributs internes par dièse (#)
        # Format V2 : Type#Name#Country#ADM1#ADM2#Lat#Long#ID#Offset
        parts = block.strip().split('#')
        if len(parts) >= 3:
            loc_type = parts[0]     # Location Type (1 = Country)
            full_name = parts[1].lower().strip() # Full Name
            country_code = parts[2] # FIPS Country Code [19, 61]

            # Application de vos filtres SQL exacts
            if loc_type == '1' and full_name == 'benin' and country_code == 'BN':
                return True
    return False

# Appliquer la fonction à la colonne 'Locations' pour créer un masque
benin_mask = gkg_df['Locations'].apply(check_geographic_precision)
# Filtrer le DataFrame pour ne garder que les lignes correspondant au Bénin
gkg_benin_df = gkg_df[benin_mask].copy().reset_index(drop=True)
gkg_benin_df.drop(columns=['GKGRECORDID'], inplace=True, errors='ignore')
print(f"✅ GKG filtré pour le Bénin : {gkg_benin_df.shape[0]:,} lignes")
# Aperçu rapide
print(gkg_benin_df[['GKGRECORDID', 'Locations']].head(10).to_string())

## Structure des datasets

In [3]:
print('\n' + '=' * 60)
print('APERÇU GÉNÉRAL GKG')
print('=' * 60)
print(f"Nombre de documents  : {gkg_df.shape[0]:,}")
print(f"Nombre de colonnes   : {gkg_df.shape[1]}")
print(f"Période couverte     : {gkg_df['Date'].min()} → {gkg_df['Date'].max()}")
print(f"Mémoire utilisée     : {gkg_df.memory_usage(deep=True).sum() / 1e6:.1f} MB")


APERÇU GÉNÉRAL GKG
Nombre de documents  : 285,043
Nombre de colonnes   : 9
Période couverte     : 20210101004500 → 20260523233000
Mémoire utilisée     : 254.8 MB


In [7]:
# 2.3 Valeurs manquantes Evénements et doublons events_df
seen = set()
clean_chunks = []

for chunk in pd.read_csv(EVENTS_DATA_PATH, chunksize=100_000):

    # drop colonnes vides
    missing_pct = chunk.isnull().mean()
    chunk = chunk.drop(columns=missing_pct[missing_pct > 0.8].index, errors="ignore")

    # déduplication locale
    chunk = chunk.drop_duplicates()

    clean_chunks.append(chunk)

events_df = pd.concat(clean_chunks, ignore_index=True)

# déduplication finale (sécurité)
events_df_no_duplicates = events_df.drop_duplicates()
events_df_no_duplicates.head()

C:\Users\dell\AppData\Local\Temp\ipykernel_5108\2984619238.py:5: DtypeWarning: Columns (0: Actor2Type3Code) have mixed types. Specify dtype option on import or set low_memory=False.
  for chunk in pd.read_csv(EVENTS_DATA_PATH, chunksize=100_000):


,GLOBALEVENTID,SQLDATE,MonthYear,Year,FractionDate,Actor1Code,Actor1Name,Actor1CountryCode,Actor1Type1Code,Actor2Code,...,Actor2Geo_FeatureID,ActionGeo_Type,ActionGeo_FullName,ActionGeo_CountryCode,ActionGeo_ADM1Code,ActionGeo_Lat,ActionGeo_Long,ActionGeo_FeatureID,DATEADDED,SOURCEURL
0,1087808447,20230305,202303,2023,2023.1781,BEN,BENIN,BEN,NaN,NaN,...,NaN,1,Benin,BN,BN,9.5,2.25,BN,20230305080000,https://newsghana.com.gh/gna-tema-and-internat...
1,1087837319,20230305,202303,2023,2023.1781,BEN,BENIN,BEN,NaN,NaN,...,NaN,1,Benin,BN,BN,9.5,2.25,BN,20230305141500,https://tribuneonlineng.com/lapo-disburses-n70...
2,1087817496,20230305,202303,2023,2023.1781,IGOCOPITP,INTERPOL,NaN,IGO,BEN,...,BN,1,Benin,BN,BN,9.5,2.25,BN,20230305101500,https://pmnewsnigeria.com/2023/03/05/interpol-...
3,1087860713,20230305,202303,2023,2023.1781,GOV,GOVERNMENT,NaN,GOV,BEN,...,BN,1,Benin,BN,BN,9.5,2.25,BN,20230305193000,https://dailypost.ng/2023/03/05/edo-lawmakers-...
4,1087860725,20230305,202303,2023,2023.1781,GOV,GOVERNMENT,NaN,GOV,LEG,...,BN,1,Benin,BN,BN,9.5,2.25,BN,20230305193000,https://dailypost.ng/2023/03/05/edo-lawmakers-...


In [3]:
# Nettoyage des doublons pour gkg_df
# s'assurer que le code d'importation des modules est exexuté

# Importer la fonction depuis scripts/data_pipeline.py
sys.path.append(str(Path().resolve().parent))
import scripts.data_pipeline as dp
import importlib
# Exécute le fichier data_pipeline.py et le recharge afin de s’assurer que les dernières modifications sont prises en compte dans la session en cours.
importlib.reload(dp)

chunks = []
for chunk in pd.read_csv(GKG_2021_2026_DATA_PATH, chunksize=100_000):

    chunk = chunk.drop_duplicates()
    parsed = chunk["TranslationInfo"].apply(dp.parse_translation_info).apply(pd.Series)
    chunk = pd.concat([chunk, parsed], axis=1)

    chunks.append(chunk)

gkg_df_processing = pd.concat(chunks, ignore_index=True)
gkg_df_processing.head()

C:\Users\dell\AppData\Local\Temp\ipykernel_8816\2237130935.py:12: DtypeWarning: Columns (0: TranslationInfo) have mixed types. Specify dtype option on import or set low_memory=False.
  for chunk in pd.read_csv(GKG_2021_2026_DATA_PATH, chunksize=100_000):
C:\Users\dell\AppData\Local\Temp\ipykernel_8816\2237130935.py:12: DtypeWarning: Columns (0: TranslationInfo) have mixed types. Specify dtype option on import or set low_memory=False.
  for chunk in pd.read_csv(GKG_2021_2026_DATA_PATH, chunksize=100_000):


,GKGRECORDID,Date,SourceCommonName,Persons,Organizations,Locations,Amounts,V2Tone,TranslationInfo,translation_source_langs
0,20251012190000-T1265,20251012190000,jagran.com,sivan a sun sharma;ashok mahato;gazipur a chau...,visa the company a operations;visa;group the c...,"4#Patna, Bihar, India#IN#IN34#25.6#85.1167#-21...","3,kilometers a distance on,57;3,month a floati...","-4.30622009569378,1.19617224880383,5.502392344...",srclc:hin;eng:GT-HIN 1.0,hin
1,20251004101500-T226,20251004101500,lapsi.al,niko peleshi;kavaja rrogozhin,us territory is us center;victory party,"4#Tirana, Tirane, Albania#AL#AL50#41.3275#19.8...",NaN,"-2.27507755946225,2.06825232678387,4.343329886...",srclc:sqi;eng:GT-SQI 1.0,sqi
2,20251004101500-T221,20251004101500,syri.net,belinda balluku;niko peleshi,you us race is party;commission to assembly;vi...,"1#Benin#BN#BN#9.5#2.25#BN;4#Tirana, Tirane, Al...",NaN,"-3.04518664047151,1.96463654223969,5.009823182...",srclc:sqi;eng:GT-SQI 1.0,sqi
3,20251004101500-T214,20251004101500,balkanweb.com,NaN,us territory is us center;do campaign corner w...,"4#Tirana, Tirane, Albania#AL#AL50#41.3275#19.8...",NaN,"-1.42011834319527,2.36686390532544,3.786982248...",srclc:sqi;eng:GT-SQI 1.0,sqi
4,20251004100000-T590,20251004100000,sot.com.al,belinda balluku;niko peleshi,you us race is party;victory party;us territor...,"4#Veliaj, Berat, Albania#AL#AL40#40.5456#20.03...",NaN,"-2.9382957884427,1.9588638589618,4.89715964740...",srclc:sqi;eng:GT-SQI 1.0,sqi


### Enregistrement des datasets nettoyés

In [ ]:
# Définir les chemins de sortie
output_dir = Path('..') / 'data' / 'processed'
output_dir.mkdir(parents=True, exist_ok=True)

events_output = output_dir / 'events_2021_2026_cleaned.csv'
gkg_output = output_dir / 'gkg_2021_2026_cleaned.csv'

# Enregistrer events_df nettoyé
events_df_no_duplicates.to_csv(events_output, index=False)
print(f"✅ Events dataset enregistré : {events_output}")
print(f"   {events_df_no_duplicates.shape[0]:,} lignes × {events_df_no_duplicates.shape[1]} colonnes")

# Enregistrer gkg_df_processing avec les colonnes de traduction
gkg_df_processing.to_csv(gkg_output, index=False)
print(f"\n✅ GKG dataset (traité) enregistré : {gkg_output}")
print(f"   {gkg_df_processing.shape[0]:,} lignes × {gkg_df_processing.shape[1]} colonnes")
print(f"   Colonnes ajoutées: translation_source_langs")


✅ GKG dataset (traité) enregistré : ..\data\processed\gkg_2021_2026_cleaned.csv
   285,043 lignes × 10 colonnes
   Colonnes ajoutées: translation_source_langs
